# UJIIndoorLoc trên Google Colab

Notebook lấy mã nền từ `main`, sau đó tự bổ sung plugin mới từ tất cả nhánh `feature/*` mà token đọc được. Menu phát hiện model và cách xử lý RSS từ các nhánh đã tải, gồm KNN/WKNN, RSS powered, CDAE-CNN và SAE/CDAE-MMJ. File đã có ở `main` không bị ghi đè; thay đổi file dùng chung cần được merge vào `main`. **Không lưu GitHub token trong notebook.** Bật GPU trước khi chạy model neural nặng. Official test chỉ dùng để báo cáo sau khi chọn model bằng internal validation.


In [ ]:
import io
import json
import shutil
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile
from pathlib import Path
from google.colab import files, userdata

REPO = "VietUniverse/ips-wifi-research"
API = f"https://api.github.com/repos/{REPO}"
root = Path("/content/repo_source")
source_cache = Path("/content/feature_sources")
try:
    token = str(userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    raise RuntimeError("Repo private: trong Colab → Secrets, thêm GITHUB_TOKEN có quyền Contents: read và Metadata: read cho VietUniverse/ips-wifi-research, bật quyền truy cập cho notebook rồi chạy lại ô này.")
headers = {"Accept": "application/vnd.github+json", "Authorization": f"Bearer {token}", "X-GitHub-Api-Version": "2022-11-28"}
token = None  # Không giữ token trong biến sau khi tạo headers

def github_get(url, timeout=120):
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.read()
    except urllib.error.HTTPError as error:
        if error.code == 401:
            raise RuntimeError("GitHub HTTP 401: token sai/hết hạn hoặc thiếu quyền Contents: read + Metadata: read. Tạo token mới, cập nhật Colab Secret GITHUB_TOKEN, rồi chạy lại ô đầu.") from None
        if error.code == 404:
            raise RuntimeError("GitHub HTTP 404: repo/nhánh không truy cập được. Kiểm tra tên repo và quyền của token với repo private.") from None
        raise RuntimeError(f"GitHub API lỗi HTTP {error.code}; kiểm tra mạng hoặc quyền token.") from None
    except urllib.error.URLError as error:
        raise RuntimeError(f"Không kết nối được GitHub: {error.reason}") from None

def extract_branch(branch, destination):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    archive = github_get(f"{API}/zipball/{branch}")
    with zipfile.ZipFile(io.BytesIO(archive)) as zipped:
        zipped.extractall(destination)
    project_files = list(destination.rglob("pyproject.toml"))
    if len(project_files) != 1:
        raise RuntimeError(f"Nhánh {branch}: cần đúng một pyproject.toml, tìm thấy {len(project_files)}")
    return project_files[0].parent

for folder in (root, source_cache):
    if folder.exists():
        shutil.rmtree(folder)
root.mkdir(parents=True)
source_cache.mkdir(parents=True)
project = extract_branch("main", root)
branch_list = json.loads(github_get(f"{API}/branches?per_page=100", timeout=30))
feature_branches = sorted(item["name"] for item in branch_list if item["name"].startswith("feature/"))
if not feature_branches:
    print("GitHub không trả về nhánh feature/*. Sẽ dùng model đã có trong main.")
print("Mã nền: main")
print("Đang quét các nhánh nhóm:", feature_branches or "(chưa có)")

def is_plugin_or_config(relative):
    parts = relative.parts
    if len(parts) == 4 and parts[:3] == ("src", "ips", "algorithms") and relative.suffix == ".py":
        return True
    if len(parts) == 5 and parts[:4] == ("src", "ips", "data", "preprocessors") and relative.suffix == ".py":
        return True
    if len(parts) == 4 and parts[:3] == ("src", "ips", "neighbors") and relative.suffix == ".py":
        return True
    if len(parts) == 4 and parts[:3] == ("src", "ips", "experiments") and parts[3].startswith("knn_") and relative.suffix == ".py":
        return True
    if len(parts) == 2 and parts[0] == "configs" and relative.suffix == ".json":
        return True
    return False

loaded, skipped = {}, []
for branch in feature_branches:
    branch_dir = source_cache / branch.replace("/", "__")
    try:
        branch_project = extract_branch(branch, branch_dir)
    except (RuntimeError, zipfile.BadZipFile) as error:
        print(f"Bỏ qua {branch}: {error}")
        continue
    count = 0
    for source in branch_project.rglob("*"):
        if not source.is_file():
            continue
        relative = source.relative_to(branch_project)
        if not is_plugin_or_config(relative):
            continue
        target = project / relative
        if target.exists():
            skipped.append(f"{branch}:{relative.as_posix()}")
            continue  # main/nhánh đã tải trước là nguồn chuẩn cho file trùng tên
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        count += 1
    loaded[branch] = count
    print(f"  {branch}: thêm {count} file plugin/config")

print("Tổng file trùng tên được giữ nguyên theo main/nhánh đầu tiên:", len(skipped))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(project)], check=True)
sys.path.insert(0, str(project / "src"))
print("Project:", project, "| Python:", sys.executable)


In [ ]:
import shutil
import urllib.request

data_dir = Path("/content/data/uji")
data_dir.mkdir(parents=True, exist_ok=True)
if not all((data_dir / name).exists() for name in ("trainingData.csv", "validationData.csv")):
    zip_path = Path("/content/ujiindoorloc.zip")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zip_path)
    extracted = Path("/content/uji_source")
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extracted)
    for name in ("trainingData.csv", "validationData.csv"):
        matches = list(extracted.rglob(name))
        if len(matches) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất {name}")
        shutil.copy2(matches[0], data_dir / name)
print("UJI data:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])


## Chọn model và tiền xử lý

Ô sau in danh sách model, tiền xử lý RSS và preset đã tải được, rồi **dừng để bạn nhập lựa chọn**. Có thể chọn nhiều model cùng họ bằng các số cách nhau dấu phẩy. KNN/WKNN và RSS powered đến từ nhánh KNN; CDAE-CNN và SAE/CDAE-MMJ đến từ các nhánh model tương ứng. Sau khi thành viên push plugin/config mới, chạy lại notebook từ ô đầu để nhận mã mới. Preset Mixer chỉ chạy với `bsrbf_kan` và `rss_standard` (520 AP).


In [ ]:
import json

options = json.loads(subprocess.check_output(
    [sys.executable, "-m", "ips.experiments.cli", "--list-options"], text=True))
presets = sorted(str(p.relative_to(project)) for p in (project / "configs").glob("*.json"))
families = {"Neural": options["models"]}
if (project / "src/ips/neighbors/__init__.py").is_file():
    neighbor_probe = subprocess.run(
        [sys.executable, "-c", "import json; from ips.neighbors import available_models; print(json.dumps(sorted(available_models())))"],
        capture_output=True, text=True)
    if neighbor_probe.returncode:
        raise RuntimeError(f"Không tải được plugin KNN: {neighbor_probe.stderr}")
    families["KNN/WKNN"] = json.loads(neighbor_probe.stdout)

def choose_one(title, values):
    if not values: raise ValueError(f'Không có lựa chọn cho {title}')
    print('\n' + title)
    for i, value in enumerate(values, 1): print(f'  {i}. {value}')
    while True:
        raw = input(f'Nhập số 1..{len(values)}: ').strip()
        if raw.isdigit() and 1 <= int(raw) <= len(values): return values[int(raw)-1]
        print('Số không hợp lệ, hãy nhập lại.')

def choose_many(title, values):
    if not values: raise ValueError(f'Không có model cho {title}')
    print('\n' + title)
    for i, value in enumerate(values, 1): print(f'  {i}. {value}')
    while True:
        raw = input('Nhập số model, ví dụ 1 hoặc 1,3: ').strip()
        try:
            indices = [int(x.strip()) for x in raw.split(',')]
            if indices and len(indices) == len(set(indices)) and all(1 <= i <= len(values) for i in indices):
                return [values[i-1] for i in indices]
        except ValueError: pass
        print('Danh sách không hợp lệ, hãy nhập lại.')

selected_family = choose_one('Chọn họ model', list(families))
selected_models = choose_many('Chọn model', families[selected_family])
selected_preprocess = choose_one('Chọn cách xử lý RSS', options['preprocessors'])
family_presets = [p for p in presets if Path(p).name.startswith('knn_') == (selected_family == 'KNN/WKNN')]
selected_preset = choose_one('Chọn preset', family_presets)
preprocess_params = json.loads(input('RSS params dạng JSON (Enter để dùng {}): ').strip() or '{}')
if not isinstance(preprocess_params, dict): raise ValueError('RSS params phải là JSON object')
print('\nĐã chọn:', selected_family, selected_models, selected_preprocess, selected_preset)


In [ ]:
import torch
from datetime import datetime, timezone

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
run_dir = Path("/content/runs") / RUN_NAME
if not selected_models or not selected_preprocess or not selected_preset:
    raise ValueError("Hãy chọn model, RSS và preset")
settings = json.loads((project / selected_preset).read_text())
print('Chạy:', selected_models, '| Tiền xử lý:', selected_preprocess, '| Preset:', selected_preset)
print('Cấu hình preset:', json.dumps(settings, ensure_ascii=False, indent=2))
print('Số dòng dữ liệu:', [(p.name, sum(1 for _ in p.open()) - 1) for p in data_dir.glob('*.csv')])
from ips.experiments.visualize import render_run, compare_runs
run_dirs = []
if selected_family == "KNN/WKNN":
    settings.update(models=selected_models, preprocess=selected_preprocess,
                    preprocess_params=preprocess_params)
    selected_config = Path("/content") / f"selected_{RUN_NAME}.json"
    selected_config.write_text(json.dumps(settings), encoding="utf-8")
    subprocess.run([sys.executable, "-u", "-m", "ips.experiments.knn_cli", "--config", str(selected_config),
                    "--data-dir", str(data_dir), "--output-dir", str(run_dir)], check=True)
    render_run(run_dir)
    run_dirs.append(run_dir)
else:
    if settings.get("with_mixer") and (selected_models != ["bsrbf_kan"] or selected_preprocess != "rss_standard"):
        raise ValueError("Preset Mixer cần model bsrbf_kan và RSS rss_standard")
    print("GPU available:", torch.cuda.is_available())
    if "bsrbf_kan" in selected_models and not torch.cuda.is_available():
        print("Cảnh báo: BSRBF-KAN trên CPU có thể rất chậm.")
    for selected_model in selected_models:
        model_dir = run_dir / selected_model
        print('Đang chạy:', selected_model)
        subprocess.run([sys.executable, "-u", "-m", "ips.experiments.cli", "--config", str(project / selected_preset),
                        "--model", selected_model, "--preprocess", selected_preprocess,
                        "--preprocess-params", json.dumps(preprocess_params),
                        "--data-dir", str(data_dir), "--output-dir", str(model_dir)], check=True)
        render_run(model_dir)
        run_dirs.append(model_dir)
    if len(run_dirs) > 1:
        compare_runs(run_dirs, labels=selected_models)
print('Kết quả:', run_dir)


In [ ]:
archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive)
files.download(archive)
